# US Flight Delays and Cancellations: Prediction Report

**Author:** Stephen Handy
**Course:** CSC381-01
**Professor:** Yunlong Wang
**Term:** Fall 2026

## Abstract

This report investigates whether 2025-2026 flight data can be used to predict flight delays or cancellations before departure. Analysis includes data, preprocessing decisions, models, evaluation results, and limitations.

## Background

Flight delays and cancellations affect passengers, airlines, airports, and connecting flights. A pre-departure prediction can help passengers assess travel risk and airlines & airports to plan staffing and operations. 

False negatives may cause travelers to miss connections or arrive unprepared. False positives may create unnecessary concern or operational cost. 

## 2. Research Question and Hypothesis

### Research Question

Can a flight's pre-departure information predict whether this flight will be delayed or cancelled?

### Hypothesis

Airline, route, airport, schedule departure time, and date features will provide predictive information. However, weather, maintenance, air-traffic conditions, and unexpected operational events will limit predictive accuracy. 

The target variable, 'disrupted', equals 1 when a flight is delayed or cancelled and 0 otherwise.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 381


All models use the same random seed and test set so that their results can be compared fairly. 

## 3. Dataset

The dataset contains US flight records from 2025-2026.
Each row represents one scheduled flight.

**Source:** https://www.kaggle.com/datasets/samuelpombourcq/us-flight-delays-and-cancellations-2025-2026
**Retrieved:** November 2026
**Target:** Whether the flight was delayed or cancelled.

In [2]:
DATA_PATH = "dataset/flights_sample.csv"

df = pd.read_csv(DATA_PATH)

display(df.head())
print("Rows and columns:", df.shape)
display(df.dtypes)


,flight_date,year,month,day_of_week,airline_code,airline,tail_number,flight_number,origin,origin_city,...,diverted,sched_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2025-01-01,2025,1,3,AA,American Airlines,N960NN,2158,SMF,"Sacramento, CA",...,0,249,221.0,202.0,1781,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,2025,1,3,B6,JetBlue Airways,N3145J,98,DEN,"Denver, CO",...,0,226,215.0,180.0,1626,0.0,0.0,0.0,0.0,96.0
2,2025-01-01,2025,1,3,AS,Alaska Airlines,N973AK,84,ANC,"Anchorage, AK",...,0,213,192.0,176.0,1448,NaN,NaN,NaN,NaN,NaN
3,2025-01-01,2025,1,3,AS,Alaska Airlines,N296AK,704,SEA,"Seattle, WA",...,0,152,128.0,110.0,867,NaN,NaN,NaN,NaN,NaN
4,2025-01-01,2025,1,3,B6,JetBlue Airways,N955JB,2169,BOS,"Boston, MA",...,0,212,204.0,184.0,1237,NaN,NaN,NaN,NaN,NaN


Rows and columns: (500000, 40)


flight_date                str
year                     int64
month                    int64
day_of_week              int64
airline_code               str
airline                    str
tail_number                str
flight_number            int64
origin                     str
origin_city                str
origin_state               str
dest                       str
dest_city                  str
dest_state                 str
sched_dep_time           int64
dep_time               float64
dep_delay              float64
dep_del15              float64
dep_time_block             str
taxi_out               float64
wheels_off             float64
wheels_on              float64
taxi_in                float64
sched_arr_time           int64
arr_time               float64
arr_delay              float64
arr_del15              float64
cancelled                int64
cancellation_code          str
cancellation_reason        str
diverted                 int64
sched_elapsed_time       int64
actual_e

In [3]:
print("Duplicate rows:", df.duplicated().sum())

missing_values = (
    df.isna()
        .sum()
        .sort_values(ascending=False)
)

display(missing_values[missing_values > 0])


Duplicate rows: 0


cancellation_reason    491116
cancellation_code      491116
late_aircraft_delay    388296
security_delay         388296
nas_delay              388296
weather_delay          388296
carrier_delay          388296
air_time                10364
actual_elapsed_time     10364
arr_del15               10364
arr_delay               10364
arr_time                 9070
taxi_in                  9070
wheels_on                9070
taxi_out                 8813
wheels_off               8813
dep_delay                8520
dep_del15                8520
dep_time                 8486
tail_number              1162
dtype: int64

## 4. Data Preparation

### Target Construction

The 'disrupted' target combines delays and cancellations.
A flight is labelled as disrupted when it satisfies the delay threshold or is cancelled.

Only columns available before departure will be used as predictors.

In [4]:
df["disrupted"] = (
    df["arr_del15"].fillna(0).eq(1)
    | df["cancelled"].eq(1)
)
df["disrupted"] = df["disrupted"].astype(int)


In [5]:
target_summary = pd.DataFrame({
    "count": df["disrupted"].value_counts(),
    "percentage": df["disrupted"].value_counts(normalize=True).mul(100).round(2),
})

display(target_summary)


,count,percentage
disrupted,,
0,379412,75.88
1,120588,24.12


In [6]:
df["sched_dep_hour"] = (df["sched_dep_time"] // 100).astype("Int64")
df["route"] = df["origin"].astype(str) + "_" + df["dest"].astype(str)


In [7]:
leakage_columns = [
    "dep_time", "dep_delay", "dep_del15", "wheels_off", "wheels_on",
    "taxi_out", "taxi_in", "arr_time", "arr_delay", "arr_del15",
    "actual_elapsed_time", "air_time", "carrier_delay", "weather_delay",
    "nas_delay", "security_delay", "late_aircraft_delay", "cancellation_code",
    "cancellation_reason", "diverted",
]

feature_columns = [
    "airline_code", "origin", "dest", "month", "day_of_week",
    "sched_dep_hour", "dep_time_block", "distance", "sched_elapsed_time",
    "route",
]

df_model = df.drop(columns=leakage_columns)
X = df_model[feature_columns]
y = df_model["disrupted"]

df_model["flight_date"] = pd.to_datetime(df_model["flight_date"])
split_date = df_model["flight_date"].quantile(0.80)
train = df_model[df_model["flight_date"] <= split_date]
test = df_model[df_model["flight_date"] > split_date]

X_train = train[feature_columns]
y_train = train["disrupted"]
X_test = test[feature_columns]
y_test = test["disrupted"]

categorical = [
    "airline_code", "origin", "dest", "month", "day_of_week",
    "dep_time_block", "route",
]
numeric = ["sched_dep_hour", "distance", "sched_elapsed_time"]

preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric),
])

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),
])
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
])

models = {
    "Baseline": baseline,
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
}

def evaluate_model(name, model):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    }

results = pd.DataFrame([
    evaluate_model(name, model) for name, model in models.items()
])
display(results)


KeyboardInterrupt: 

## 5. Feature Selection and Leakage Prevention

The prediction is made before departure. Therefore, actual departure, arrival, delay-cause, cancellation-reason, and diversion fields are excluded. The chronological 80/20 split also ensures that later dates are evaluated as future observations.

## 6. Model Results

The majority-class baseline is compared with logistic regression and random forest using the same test period. Because disruptions are less common than non-disruptions, recall and F1 score are more informative than accuracy alone.

In [ ]:
selected_model = random_forest_model
selected_predictions = selected_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    selected_predictions,
    display_labels=["Not Disrupted", "Disrupted"],
    cmap="Blues",
)
plt.title("Confusion Matrix: Random Forest")
plt.show()

feature_names = selected_model.named_steps["preprocessor"].get_feature_names_out()
importance_table = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": selected_model.named_steps["model"].feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .head(20)
)

display(importance_table)
sns.barplot(data=importance_table, x="importance", y="feature")
plt.title("Top Predictive Features")
plt.xlabel("Random-Forest Feature Importance")
plt.ylabel("Feature")
plt.show()


## 7. Limitations

The dataset does not provide complete pre-departure information about weather, maintenance, air traffic, or unexpected operational events. The sample covers reporting carriers and may not represent every US airline. Feature importance indicates predictive association, not causation.

## 8. Conclusion

The results should be summarized using the model-comparison table above. State which model performed best according to F1 or recall, whether it improved over the baseline, and whether the results support the hypothesis. Replace this paragraph with the observed metric values after executing the notebook.